In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
import os
import transformers
import numpy as np

# Print transformers version
print(f"Using transformers version: {transformers.__version__}")

# Clean text
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|\#\w+', '', text)
    text = text.lower().strip()
    return text

# Load and oversample data
def load_data(file_path):
    df = pd.read_csv(file_path)
    df = df[['tweet', 'class']].rename(columns={'tweet': 'text', 'class': 'label'})
    df['text'] = df['text'].apply(clean_text)
    df['label'] = df['label'].astype(int)

    # Oversample hate (class 0) and offensive (class 1)
    hate_df = df[df['label'] == 0]
    offensive_df = df[df['label'] == 1]
    hate_oversample_factor = 3  # Repeat hate 3x
    offensive_oversample_factor = 1  # Repeat offensive 1x (total 2x)
    df_oversampled = pd.concat([df] + [hate_df] * (hate_oversample_factor - 1) + [offensive_df] * offensive_oversample_factor, ignore_index=True)

    # Print dataset size and class distribution
    print(f"Dataset size: {len(df_oversampled)} examples")
    print("Class distribution:")
    print(df_oversampled['label'].value_counts(normalize=True))

    return df_oversampled

# Dataset class
class HateSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Custom trainer with class weights
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        class_weights = torch.tensor([3.5, 2.5, 0.2]).to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Train model
def train_model(train_dataset, val_dataset, output_dir):
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=3, dropout=0.4
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        warmup_steps=500,
        weight_decay=0.02,
        learning_rate=3e-5,
        logging_dir='./logs',
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer

# Compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Predict new text
def predict_text(text, model, tokenizer, max_len=128):
    model.eval()
    text = clean_text(text)
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    model = model.to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = torch.argmax(logits, dim=1).item()

    label_map = {0: 'hate', 1: 'offensive', 2: 'neutral'}
    print(f"Probabilities: hate={probs[0]:.2f}, offensive={probs[1]:.2f}, neutral={probs[2]:.2f}")
    return label_map[pred]

def main():
    data_path = '/content/labeled_data.csv'
    output_dir = '/content/model_output'

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("Loading data...")
    df = load_data(data_path)
    texts = df['text'].values
    labels = df['label'].values

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.2, random_state=42
    )

    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

    train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer)
    val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer)

    print("Training model...")
    trainer = train_model(train_dataset, val_dataset, output_dir)

    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    model = DistilBertForSequenceClassification.from_pretrained(output_dir)
    print("\nTesting predictions:")
    test_texts = [
        "I hate everyone in this group!",
        "You're such a loser, go away.",
        "Have a nice day everyone!"
    ]

    for text in test_texts:
        pred = predict_text(text, model, tokenizer)
        print(f"Text: {text}")
        print(f"Prediction: {pred}\n")

if __name__ == "__main__":
    main()

Using transformers version: 4.51.3
Loading data...
Dataset size: 46833 examples
Class distribution:
label
1    0.819508
0    0.091602
2    0.088890
Name: proportion, dtype: float64
Training model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.268000,0.314208,0.880218,0.874572,0.881939,0.880218
2,0.226700,0.228042,0.899114,0.898462,0.906322,0.899114
3,0.215800,0.221393,0.898900,0.899785,0.910784,0.898900



Testing predictions:
Probabilities: hate=0.78, offensive=0.20, neutral=0.02
Text: I hate everyone in this group!
Prediction: hate

Probabilities: hate=0.32, offensive=0.64, neutral=0.04
Text: You're such a loser, go away.
Prediction: offensive

Probabilities: hate=0.01, offensive=0.36, neutral=0.62
Text: Have a nice day everyone!
Prediction: neutral



In [3]:
# Clean text
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|\#\w+', '', text)
    text = text.lower().strip()
    return text

# Predict text with probabilities
def predict_text(text, model, tokenizer, max_len=128):
    model.eval()
    text = clean_text(text)
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    model = model.to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = torch.argmax(logits, dim=1).item()

    label_map = {0: 'hate', 1: 'offensive', 2: 'neutral'}
    print(f"Probabilities: hate={probs[0]:.2f}, offensive={probs[1]:.2f}, neutral={probs[2]:.2f}")
    return label_map[pred]

# Load model and tokenizer
print("Loading model and tokenizer...")
model = DistilBertForSequenceClassification.from_pretrained('/content/model_output')
tokenizer = DistilBertTokenizer.from_pretrained('/content/model_output')

# Test examples
print("Testing predictions:")
test_texts = [
    "I hate everyone in this group!",
    "You're such a loser, go away.",
    "Have a nice day everyone!",
    "This group is full of disgusting people, I despise them!",
    "You’re a complete idiot, get lost!",
    "I love hanging out with my friends, such a great day!",
    "I loathe this community, they’re all trash.",
    "Why are you so stupid? Just leave!",
    "Thanks for the help, really appreciate it!",
    "I hate this place but the food is okay.",
    "You’re such a jerk, nobody likes you.",
    "Everyone here is so annoying, I can’t stand them!",
    "Let’s all have a fun time together!"
]

for text in test_texts:
    pred = predict_text(text, model, tokenizer)
    print(f"Text: {text}")
    print(f"Prediction: {pred}\n")

Loading model and tokenizer...
Testing predictions:
Probabilities: hate=0.78, offensive=0.20, neutral=0.02
Text: I hate everyone in this group!
Prediction: hate

Probabilities: hate=0.32, offensive=0.64, neutral=0.04
Text: You're such a loser, go away.
Prediction: offensive

Probabilities: hate=0.01, offensive=0.36, neutral=0.62
Text: Have a nice day everyone!
Prediction: neutral

Probabilities: hate=0.98, offensive=0.01, neutral=0.00
Text: This group is full of disgusting people, I despise them!
Prediction: hate

Probabilities: hate=0.09, offensive=0.59, neutral=0.32
Text: You’re a complete idiot, get lost!
Prediction: offensive

Probabilities: hate=0.02, offensive=0.25, neutral=0.73
Text: I love hanging out with my friends, such a great day!
Prediction: neutral

Probabilities: hate=0.98, offensive=0.02, neutral=0.00
Text: I loathe this community, they’re all trash.
Prediction: hate

Probabilities: hate=0.07, offensive=0.90, neutral=0.03
Text: Why are you so stupid? Just leave!
Predic